# 07 — End-to-end pipeline with the agent stub (Checkpoint 1)

Until 18.10.2026 the agent layer is a **stub**: it takes the output of the frozen model for one study
(`ModelOutput`) and returns one `AgentReport`. It has **no LLM, no guardrails and no tools**; it only restates
the model flag against the frozen threshold. The real agent (Checkpoint 2, after 18.10) will replace
`agent_stub` without changing this contract.

This notebook:
1. runs every internal-validation study of the pipeline model through the stub and writes `case_reports.jsonl`;
2. runs a live check, image → frozen model → stub, and confirms it matches the saved scores
   (`LIVE_CHECK_STUDIES`: `"all"` = all validation studies, the acceptance run; a number, e.g. 20 = quick check);
3. demonstrates one study end to end (`DEMO_STUDY_ID`; radiographs only if `SHOW_IMAGES = True`).

Run it on a **GPU runtime (T4)**: the saved Model B scores were computed on GPU with mixed precision, and a CPU
run differs by more than the 0.002 tolerance of the live check.

Inputs (read-only): `operating_policy.json` from Notebook 06a and the saved validation predictions.
Outputs: a new folder `DATA/RUNS_ARTEM/07_AGENT_STUB_PIPELINE/run_…/`.

In [ ]:
import hashlib
import json
import random
import sys

import numpy as np
import pandas as pd
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("Device:", DEVICE)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
from pathlib import Path
from datetime import datetime, timezone

# The team DATA folder is shared from the project owner's Drive. Members see it
# either through a "DATA" shortcut in My Drive or through the shared
# "Final Project DS23" shortcut. Both resolve to the same physical folder.
DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_DIR_CANDIDATES = [
    DRIVE_ROOT / "DATA",
    DRIVE_ROOT
    / "Final Project DS23"
    / "Agentic AI-Assisted Detection of Musculoskeletal Abnormalities in Emergency X-rays"
    / "DATA",
]


def resolve_data_dir(candidates=DATA_DIR_CANDIDATES):
    for candidate in candidates:
        if (candidate / "MURA-v1.1_files.zip").exists() and (candidate / "WRIST_SPLITS").is_dir():
            return candidate
    raise FileNotFoundError(
        "Team DATA folder not found. Add a 'DATA' shortcut to My Drive or check "
        "that 'Final Project DS23' is in My Drive. Tried:\n"
        + "\n".join(str(c) for c in candidates)
    )


DATA_DIR = resolve_data_dir()
RAW_ZIP = DATA_DIR / "MURA-v1.1_files.zip"
SPLITS_DIR = DATA_DIR / "WRIST_SPLITS"            # read-only here (Notebook 01 output)
PREPROCESSING_DIR = DATA_DIR / "PREPROCESSED_WRIST"  # read-only here (Notebook 02 output)
MODEL_A_DIR = DATA_DIR / "BASELINE_MODEL_A"        # read-only here (Notebook 03 output)

# Shared-folder rule: this notebook never modifies or replaces existing team
# files. Everything it writes goes to a NEW run folder under RUNS_ARTEM.
RUN_ROOT = DATA_DIR / "RUNS_ARTEM"
STAGE_DIR = RUN_ROOT / "07_AGENT_STUB_PIPELINE"


def new_run_dir(stage_dir):
    run_id = datetime.now(timezone.utc).strftime("run_%Y%m%dT%H%M%SZ")
    run_dir = stage_dir / run_id
    run_dir.mkdir(parents=True, exist_ok=False)
    return run_dir


def out_path(run_dir, name):
    """Return a path inside this run's own folder; refuse anything else."""
    path = run_dir / name
    assert str(path).startswith(str(run_dir) + "/"), path
    assert not path.exists(), f"Refusing to overwrite: {path}"
    return path


print("Team DATA folder:", DATA_DIR)
print("Writes only under:", STAGE_DIR)

## The stub module

This cell is an exact copy of `src/pipeline/agent_stub.py` in the repository (Colab runs without the repo).
`tests/test_agent_stub.py` fails if the two ever differ.

In [ ]:
%%writefile agent_stub.py
"""Placeholder for the agent layer (Checkpoint 1, until 18.10.2026).

The stub fixes the contract between the frozen image model and the future agent
layer: it takes one ``ModelOutput`` per study and returns one ``AgentReport``.
It has no LLM, no guardrails and no tools. It only restates the model flag
against the frozen threshold, so the pipeline can run end to end today and the
real agent (Checkpoint 2) can replace it without changing the interface.
"""
from __future__ import annotations

import json
import math
import uuid
from dataclasses import asdict, dataclass, field
from typing import Literal, Optional, Tuple

STUB_LIMITATION = "agent layer not implemented: stub"
CONTRACT_VERSION = "1.0"

ModelFlag = Literal["ABOVE_THRESHOLD", "BELOW_THRESHOLD"]
AgentStatus = Literal["STUB"]


def _check_probability(name: str, value: float) -> float:
    value = float(value)
    if not math.isfinite(value) or not 0.0 <= value <= 1.0:
        raise ValueError(f"{name} must be a finite number in [0, 1], got {value!r}")
    return value


@dataclass(frozen=True)
class ModelOutput:
    """Stub input: the output of the frozen model for one study."""

    study_id: str
    model_version: str  # e.g. "model_a@<sha256[:12]>" or "model_b@<sha256[:12]>"
    study_probability: float
    threshold: float  # frozen operating threshold from operating_policy.json
    view_probabilities: Optional[Tuple[float, ...]] = None  # None for Model A

    def __post_init__(self) -> None:
        if not str(self.study_id).strip():
            raise ValueError("study_id must be a non-empty string")
        if not str(self.model_version).strip():
            raise ValueError("model_version must be a non-empty string")
        object.__setattr__(self, "study_probability", _check_probability("study_probability", self.study_probability))
        object.__setattr__(self, "threshold", _check_probability("threshold", self.threshold))
        if self.view_probabilities is not None:
            views = tuple(_check_probability("view_probability", v) for v in self.view_probabilities)
            if not views:
                raise ValueError("view_probabilities must be None or non-empty")
            object.__setattr__(self, "view_probabilities", views)


@dataclass(frozen=True)
class AgentReport:
    """Stub output: one report per study."""

    case_id: str
    model_flag: ModelFlag
    agent_status: AgentStatus  # at Checkpoint 2 this field carries the agent's decision
    evidence: ModelOutput  # the input, unchanged
    limitations: Tuple[str, ...] = field(default=(STUB_LIMITATION,))
    trace_id: str = ""
    contract_version: str = CONTRACT_VERSION

    def to_dict(self) -> dict:
        record = asdict(self)
        evidence = record["evidence"]
        if evidence["view_probabilities"] is not None:
            evidence["view_probabilities"] = list(evidence["view_probabilities"])
        record["limitations"] = list(record["limitations"])
        return record

    def to_json(self) -> str:
        return json.dumps(self.to_dict(), ensure_ascii=False, sort_keys=True)


def agent_stub(x: ModelOutput, *, trace_id: Optional[str] = None) -> AgentReport:
    """Return the stub report for one study. No LLM, no guardrails, no tools.

    The flag uses the same rule as the evaluation notebooks:
    probability >= threshold -> ABOVE_THRESHOLD (flag for specialist review).
    """
    if not isinstance(x, ModelOutput):
        raise TypeError("agent_stub expects a ModelOutput")
    flag: ModelFlag = "ABOVE_THRESHOLD" if x.study_probability >= x.threshold else "BELOW_THRESHOLD"
    return AgentReport(
        case_id=x.study_id,
        model_flag=flag,
        agent_status="STUB",
        evidence=x,
        limitations=(STUB_LIMITATION,),
        trace_id=trace_id if trace_id is not None else uuid.uuid4().hex,
    )

In [ ]:
sys.path.insert(0, "/content")
import importlib

import agent_stub as stub_module

importlib.reload(stub_module)
from agent_stub import CONTRACT_VERSION, STUB_LIMITATION, ModelOutput, agent_stub

POLICY_RUN_ID = "run_20261017T105209Z"  # frozen by Notebook 06a on 17.10.2026 (pipeline model B, seed 7)
POLICY_PATH = RUN_ROOT / "06A_OPERATING_POLICY" / POLICY_RUN_ID / "operating_policy.json"
policy = json.loads(POLICY_PATH.read_text())
PIPELINE_MODEL = policy["pipeline_model"]
frozen = policy["models"][PIPELINE_MODEL]
THRESHOLD = float(frozen["primary_threshold"])
MODEL_VERSION = frozen["model_version"]

# Run settings
LIVE_CHECK_STUDIES = "all"  # "all" for the acceptance run; a number (e.g. 20) for a quick check (fixed seed)
DEMO_STUDY_ID = None        # None: one fixed study chosen with the seed; or a study_id from internal validation
SHOW_IMAGES = False         # True only for a live demonstration; keep False in any output that is committed
assert LIVE_CHECK_STUDIES == "all" or int(LIVE_CHECK_STUDIES) > 0, "LIVE_CHECK_STUDIES: 'all' or a positive number"

RUN_DIR = new_run_dir(STAGE_DIR)
print("Pipeline model:", PIPELINE_MODEL, "|", MODEL_VERSION, "| frozen threshold:", round(THRESHOLD, 6))
print("This run writes to:", RUN_DIR)


def sha256_of(path, chunk=1 << 20):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()

## 1. All internal-validation studies → stub → `case_reports.jsonl`

In [ ]:
if PIPELINE_MODEL == "A":
    predictions_path = MODEL_A_DIR / "validation_predictions.csv"
    view_predictions = None
else:
    model_b_dir = RUN_ROOT / "05_MODEL_B_FINETUNE" / frozen["run_id"]
    predictions_path = model_b_dir / "validation_predictions.csv"
    view_predictions = pd.read_csv(model_b_dir / "validation_view_predictions.csv")
assert sha256_of(predictions_path) == frozen["validation_predictions_sha256"], "Predictions changed after freezing"
predictions = pd.read_csv(predictions_path)

views_by_study = {}
if view_predictions is not None:
    for study_id, group in view_predictions.sort_values("view_index").groupby("study_id"):
        views_by_study[str(study_id)] = tuple(group["view_probability"].astype(float))

reports = []
for row in predictions.itertuples(index=False):
    model_output = ModelOutput(
        study_id=str(row.study_id),
        model_version=MODEL_VERSION,
        study_probability=float(row.abnormal_probability),
        threshold=THRESHOLD,
        view_probabilities=views_by_study.get(str(row.study_id)),
    )
    reports.append(agent_stub(model_output, trace_id=f"{RUN_DIR.name}:{row.study_id}"))

# Contract checks
assert len(reports) == len(predictions) == predictions["study_id"].nunique()
assert all(r.agent_status == "STUB" and r.limitations == (STUB_LIMITATION,) for r in reports)
expected_flags = (predictions["abnormal_probability"] >= THRESHOLD).to_numpy()
assert np.array_equal(np.array([r.model_flag == "ABOVE_THRESHOLD" for r in reports]), expected_flags)
if view_predictions is not None:
    assert all(r.evidence.view_probabilities is not None for r in reports)

reports_path = out_path(RUN_DIR, "case_reports.jsonl")
with open(reports_path, "w", encoding="utf-8") as handle:
    for report in reports:
        handle.write(report.to_json() + "\n")

flag_counts = pd.Series([r.model_flag for r in reports]).value_counts().to_dict()
labels = predictions["label"].to_numpy(dtype=int)
print("Reports written:", len(reports), "->", reports_path.name)
print("Flags:", flag_counts)
print("Missed abnormal studies (BELOW_THRESHOLD with label 1):", int(((~expected_flags) & (labels == 1)).sum()))
# Example of the AgentReport contract; study and trace identifiers are masked so that
# no per-study data ends up in the committed notebook output.
example_text = json.dumps(reports[0].to_dict(), indent=2)
for identifier in (reports[0].trace_id, reports[0].case_id):  # trace_id contains the study id
    example_text = example_text.replace(str(identifier), "<masked>")
assert reports[0].case_id not in example_text
print("Example report (identifiers masked):")
print(example_text[:900])

## 2. Live end-to-end check: image → frozen model → stub

Recomputes the scores from the radiographs and compares them with the saved ones. `LIVE_CHECK_STUDIES = "all"` checks every validation study; a number checks a fixed random subset (seed 42), which catches systematic errors (wrong weights, preprocessing or hardware) but can miss problems in single studies.

In [ ]:
from PIL import Image, ImageOps
from torchvision import transforms
from torchvision.transforms import InterpolationMode


class PadToSquare:
    """Same padding rule as Notebooks 02 and 03 (black border, centred)."""

    def __call__(self, image):
        width, height = image.size
        side = max(width, height)
        left = (side - width) // 2
        top = (side - height) // 2
        right = side - width - left
        bottom = side - height - top
        return ImageOps.expand(image, border=(left, top, right, bottom), fill=0)


IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)
IMAGE_SIZE = 224


def _to_gray(image):
    return image.convert("L")


def _to_rgb(image):
    return image.convert("RGB")


# Identical to the evaluation transform of Notebook 03.
evaluation_transform = transforms.Compose([
    transforms.Lambda(_to_gray),
    PadToSquare(),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE), interpolation=InterpolationMode.BILINEAR, antialias=True),
    transforms.Lambda(_to_rgb),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

In [ ]:
import shutil
import time
import zipfile

LOCAL_ROOT = Path("/content/mura_local")
LOCAL_ZIP = LOCAL_ROOT / "MURA-v1.1_files.zip"
LOCAL_IMAGES = LOCAL_ROOT / "images"
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)


def stage_zip_locally():
    """Copy the 3.4 GB archive from Drive to the Colab disk once (read-only use)."""
    if LOCAL_ZIP.exists() and LOCAL_ZIP.stat().st_size == RAW_ZIP.stat().st_size:
        print("Local ZIP already present.")
        return
    started = time.time()
    shutil.copyfile(RAW_ZIP, LOCAL_ZIP)
    assert LOCAL_ZIP.stat().st_size == RAW_ZIP.stat().st_size
    print(f"Copied ZIP to local disk in {(time.time() - started) / 60:.1f} min")


def resolve_zip_member(value, members):
    normalized = str(value).replace("\\", "/").lstrip("/")
    for candidate in (normalized, "MURA-v1.1/" + normalized):
        if candidate in members:
            return candidate
    raise KeyError(f"Image path not found in ZIP: {value}")


def extract_members(paths):
    """Extract the listed images to local disk; return {manifest path: local file}."""
    with zipfile.ZipFile(LOCAL_ZIP) as archive:
        members = set(archive.namelist())
        mapping = {}
        for value in paths:
            member = resolve_zip_member(value, members)
            target = LOCAL_IMAGES / member
            if not target.exists():
                target.parent.mkdir(parents=True, exist_ok=True)
                with archive.open(member) as source, open(target, "wb") as sink:
                    shutil.copyfileobj(source, sink)
            mapping[value] = target
    return mapping

In [ ]:
import numpy as np
import torch
from torch.utils.data import Dataset


def load_image(path):
    with Image.open(path) as image:
        image.load()
        return image.copy()


class StudyDataset(Dataset):
    """One item per study: all its views stacked as [V, 3, 224, 224]."""

    def __init__(self, views, local_paths, transform, path_column):
        self.transform = transform
        self.studies = []
        for study_uid, group in views.groupby("study_uid", sort=True):
            assert group["patient_id"].nunique() == 1
            assert group["label_binary"].nunique() == 1
            self.studies.append({
                "study_id": str(study_uid),
                "patient_id": str(group["patient_id"].iloc[0]),
                "label": int(group["label_binary"].iloc[0]),
                "view_paths": [str(p) for p in group[path_column]],
                "local_paths": [local_paths[p] for p in group[path_column]],
            })

    def __len__(self):
        return len(self.studies)

    def __getitem__(self, index):
        study = self.studies[index]
        images = torch.stack([self.transform(load_image(p)) for p in study["local_paths"]])
        return {
            "images": images,
            "label": float(study["label"]),
            "study_id": study["study_id"],
            "patient_id": study["patient_id"],
            "index": index,
        }


def collate_studies(batch):
    max_views = max(item["images"].shape[0] for item in batch)
    _, channels, height, width = batch[0]["images"].shape
    images = torch.zeros((len(batch), max_views, channels, height, width))
    mask = torch.zeros((len(batch), max_views), dtype=torch.bool)
    for row, item in enumerate(batch):
        views = item["images"].shape[0]
        images[row, :views] = item["images"]
        mask[row, :views] = True
    return {
        "images": images,
        "mask": mask,
        "labels": torch.tensor([item["label"] for item in batch], dtype=torch.float32),
        "study_ids": [item["study_id"] for item in batch],
        "patient_ids": [item["patient_id"] for item in batch],
        "indices": [item["index"] for item in batch],
    }

In [ ]:
import torch.nn as nn
from torchvision import models


def build_model_b(pretrained=True):
    weights = models.ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
    model = models.resnet18(weights=weights)
    model.fc = nn.Linear(model.fc.in_features, 1)  # one logit per radiograph
    return model


def study_forward(model, images, mask, use_amp):
    """Per-view logits -> probabilities -> mean and max per study (padding ignored)."""
    flat = images[mask]
    with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=use_amp):
        logits = model(flat).squeeze(1)
    view_probs_flat = torch.sigmoid(logits.float())
    view_probs = torch.zeros(mask.shape, device=images.device)
    view_probs[mask] = view_probs_flat
    counts = mask.sum(dim=1).clamp(min=1)
    mean_prob = (view_probs * mask).sum(dim=1) / counts
    max_prob = view_probs.masked_fill(~mask, -1.0).max(dim=1).values
    return mean_prob, max_prob, view_probs

In [ ]:
import joblib
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import models

validation_manifest = pd.read_csv(PREPROCESSING_DIR / "wrist_internal_val_clean.csv")
PATH_CANDIDATES = ("image_path", "relative_path", "path", "file_name")
PATH_COLUMN = next(c for c in PATH_CANDIDATES if c in validation_manifest.columns)
all_study_ids = sorted(predictions["study_id"].astype(str).unique())
if LIVE_CHECK_STUDIES == "all":
    live_studies = all_study_ids
else:
    rng = np.random.default_rng(SEED)
    size = min(int(LIVE_CHECK_STUDIES), len(all_study_ids))
    live_studies = sorted(rng.choice(all_study_ids, size=size, replace=False))
print("Live check:", len(live_studies), "of", len(all_study_ids), "validation studies")
live_views = validation_manifest[validation_manifest["study_uid"].astype(str).isin(live_studies)].copy()

stage_zip_locally()
live_paths = extract_members(live_views[PATH_COLUMN].tolist())
live_loader = DataLoader(StudyDataset(live_views, live_paths, evaluation_transform, PATH_COLUMN),
                         batch_size=8, shuffle=False, collate_fn=collate_studies)

ARTIFACT = DATA_DIR / frozen["artifact"]  # stored relative to DATA (absolute in older policies: still works)
assert sha256_of(ARTIFACT) == frozen["artifact_sha256"], "Model artifact changed after freezing"
if PIPELINE_MODEL == "A":
    encoder = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    encoder.fc = nn.Identity()
    encoder = encoder.to(DEVICE).eval()
    classifier = joblib.load(ARTIFACT)
else:
    model_b = build_model_b(pretrained=False).to(DEVICE)
    model_b.load_state_dict(torch.load(ARTIFACT, map_location=DEVICE)["model_state_dict"])
    model_b.eval()


@torch.no_grad()
def live_model_outputs(loader):
    outputs = []
    for batch in loader:
        images, mask = batch["images"].to(DEVICE), batch["mask"].to(DEVICE)
        counts = mask.sum(dim=1).tolist()
        if PIPELINE_MODEL == "A":
            embeddings = encoder(images[mask]).float().cpu().numpy()
            start = 0
            for i, index in enumerate(batch["indices"]):
                pooled = embeddings[start:start + counts[i]].mean(axis=0, keepdims=True)
                start += counts[i]
                study_id = loader.dataset.studies[index]["study_id"]
                outputs.append(ModelOutput(study_id, MODEL_VERSION, float(classifier.predict_proba(pooled)[0, 1]), THRESHOLD))
        else:
            mean_prob, max_prob, view_probs = study_forward(model_b, images, mask, USE_AMP)
            chosen = mean_prob if frozen["aggregation"] == "mean" else max_prob
            for i, index in enumerate(batch["indices"]):
                study_id = loader.dataset.studies[index]["study_id"]
                views = tuple(float(v) for v in view_probs[i, : counts[i]].tolist())
                outputs.append(ModelOutput(study_id, MODEL_VERSION, float(chosen[i]), THRESHOLD, views))
    return outputs


saved = predictions.set_index(predictions["study_id"].astype(str))
live_rows = []
for model_output in live_model_outputs(live_loader):
    report = agent_stub(model_output, trace_id=f"{RUN_DIR.name}:live:{model_output.study_id}")
    saved_probability = float(saved.loc[model_output.study_id, "abnormal_probability"])
    live_rows.append({
        "study_id": model_output.study_id,
        "live_probability": model_output.study_probability,
        "saved_probability": saved_probability,
        "abs_difference": abs(model_output.study_probability - saved_probability),
        "live_flag": report.model_flag,
        "saved_flag": "ABOVE_THRESHOLD" if saved_probability >= THRESHOLD else "BELOW_THRESHOLD",
    })
live_check = pd.DataFrame(live_rows)
TOLERANCE = 2e-3
assert len(live_check) == len(live_studies)
assert live_check["abs_difference"].max() < TOLERANCE, "Live scores differ from saved validation scores"
# A flag may legitimately flip only when the saved score lies within the tolerance of the threshold.
flag_mismatch = live_check["live_flag"] != live_check["saved_flag"]
near_threshold = (live_check["saved_probability"] - THRESHOLD).abs() < TOLERANCE
assert not (flag_mismatch & ~near_threshold).any(), "Live flag differs from the saved flag away from the threshold"
live_check.to_csv(out_path(RUN_DIR, "live_end_to_end_check.csv"), index=False)  # per-study rows: Drive only
print(
    f"Live end-to-end check passed: {len(live_check)} studies | max |Δp| = {live_check['abs_difference'].max():.2e} "
    f"| flags identical: {int((~flag_mismatch).sum())}/{len(live_check)} | flips near the threshold: {int(flag_mismatch.sum())}"
)

## 3. Demonstration: one study end to end

Radiographs of one study → frozen model → stub → `AgentReport`. Identifiers are masked in the printed output. Set `SHOW_IMAGES = True` only for a live demonstration in Colab and do not commit that output (README: no raw images in the repository).

In [ ]:
import matplotlib.pyplot as plt

if DEMO_STUDY_ID is None:
    demo_id = str(np.random.default_rng(SEED + 1).choice(all_study_ids))
else:
    demo_id = str(DEMO_STUDY_ID)
    assert demo_id in set(all_study_ids), "DEMO_STUDY_ID is not an internal-validation study"

demo_views = validation_manifest[validation_manifest["study_uid"].astype(str) == demo_id].copy()
demo_paths = extract_members(demo_views[PATH_COLUMN].tolist())
demo_dataset = StudyDataset(demo_views, demo_paths, evaluation_transform, PATH_COLUMN)
demo_loader = DataLoader(demo_dataset, batch_size=1, shuffle=False, collate_fn=collate_studies)
demo_output = live_model_outputs(demo_loader)[0]
demo_report = agent_stub(demo_output, trace_id=f"{RUN_DIR.name}:demo:{demo_id}")
demo_saved = float(saved.loc[demo_id, "abnormal_probability"])
demo_label = int(saved.loc[demo_id, "label"])

print("1. Input:", len(demo_dataset.studies[0]["local_paths"]), "radiograph(s) of one study (identifier masked)")
if demo_output.view_probabilities is not None:
    print("2. Frozen model, score per view:", ", ".join(f"{p:.3f}" for p in demo_output.view_probabilities))
    print(f"   Study score ({frozen['aggregation']} of the views): {demo_output.study_probability:.3f}")
else:
    print(f"2. Frozen model (Model A, mean embedding of the views): study score {demo_output.study_probability:.3f}")
print(f"   Saved validation score for this study: {demo_saved:.3f} (|Δ| = {abs(demo_saved - demo_output.study_probability):.1e})")
print(f"3. Stub: score {demo_output.study_probability:.3f} vs frozen threshold {THRESHOLD:.4f} -> {demo_report.model_flag}")
print("   MURA study label, for reference only:", "ABNORMAL" if demo_label == 1 else "NORMAL")

demo_text = json.dumps(demo_report.to_dict(), indent=2)
for identifier in (demo_report.trace_id, demo_report.case_id):
    demo_text = demo_text.replace(str(identifier), "<masked>")
assert demo_id not in demo_text
print("4. AgentReport (identifiers masked):")
print(demo_text)
out_path(RUN_DIR, "demo_case_report.json").write_text(demo_report.to_json(), encoding="utf-8")  # Drive only

if SHOW_IMAGES:
    local = demo_dataset.studies[0]["local_paths"]
    fig, axes = plt.subplots(1, len(local), figsize=(3.2 * len(local), 3.4), squeeze=False)
    scores = demo_output.view_probabilities or [None] * len(local)
    for axis, path, score in zip(axes[0], local, scores):
        axis.imshow(load_image(path).convert("L"), cmap="gray")
        axis.set_title("view score " + (f"{score:.3f}" if score is not None else "n/a"), fontsize=9)
        axis.axis("off")
    plt.suptitle(f"Demonstration study: {demo_report.model_flag} (threshold {THRESHOLD:.4f})")
    plt.tight_layout()
    plt.show()
    print("SHOW_IMAGES is True: clear this output before saving the notebook to GitHub.")

In [ ]:
run_summary = {
    "stage": "Checkpoint 1 end-to-end run with agent stub",
    "contract_version": CONTRACT_VERSION,
    "agent": "agent_stub (no LLM, no guardrails, no tools)",
    "policy_run_id": POLICY_RUN_ID,
    "policy_sha256": sha256_of(POLICY_PATH),
    "pipeline_model": PIPELINE_MODEL,
    "model_version": MODEL_VERSION,
    "threshold": THRESHOLD,
    "validation_studies_reported": len(reports),
    "flag_counts": flag_counts,
    "live_check_setting": str(LIVE_CHECK_STUDIES),
    "live_check_studies": int(len(live_check)),
    "live_check_max_abs_difference": float(live_check["abs_difference"].max()),
    "demo_model_flag": demo_report.model_flag,
    "case_reports": reports_path.relative_to(DATA_DIR).as_posix(),
    "official_mura_valid_read": False,
}
out_path(RUN_DIR, "pipeline_run_summary.json").write_text(json.dumps(run_summary, indent=2), encoding="utf-8")
display(pd.Series(run_summary))